# Order parameter distribution: Pure DOPC vs DOPC/CHOL = 60/40

Histogram of the chain order parameter averaged over carbons C2-C17, one value per chain per frame (raw, no time averaging),
both leaflets pooled. Similar to Fig. 4 in Walter et al., PCCP 2020, 22, 19147.

The order parameters are already calculated: `heavy_data_NO_GIT/dataframes/sym/*_op_ft_raw.csv` (made by `training_df_generation`).
Each CSV has one row per chain per frame (lipid by lipid, 4000 frames each); columns C1-C16 are carbons C2-C17.

Run the four cells in order.

In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

# ---- 1. import packages ----
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

In [ ]:
# ---- 2. file paths and settings ----
data_dir = "../heavy_data_NO_GIT/dataframes/sym"
out_dir  = "../heavy_data_NO_GIT/results/s_avg_distributions"
os.makedirs(out_dir, exist_ok=True)

# file prefix -> legend label
systems = {"pure_dopc": "Pure DOPC",
           "dopc_60":   "DOPC/CHOL = 60/40"}
colors  = {"pure_dopc": "green", "dopc_60": "red"}

chains   = ["sn1", "sn2"]
leaflets = ["lower", "upper"]
carbon_columns = [f"C{i}" for i in range(1, 17)]   # C1..C16 in the CSV = carbons C2..C17

bins = np.arange(-0.6, 0.8 + 0.01, 0.01)           # histogram bins, width 0.01
x_limits = (-0.2, 0.4)                              # x axis range of the plot

In [ ]:
# ---- 3. averaged order parameter per chain, and histograms ----
# S_avg = mean of the 16 carbon order parameters of one chain in one frame
counts = {}     # (system, chain) -> number of chains in each bin

for chain in chains:
    for system in systems:
        s_avg = []
        for leaflet in leaflets:
            file = os.path.join(data_dir, f"{system}_{chain}_{leaflet}_op_ft_raw.csv")
            ops = pd.read_csv(file, usecols=carbon_columns)
            s_avg.append(ops.mean(axis=1).to_numpy())          # average over carbons, row by row
        s_avg = np.concatenate(s_avg)                          # both leaflets together
        counts[(system, chain)] = np.histogram(s_avg, bins=bins)[0]
        print(f"{chain} {systems[system]:18s}: {s_avg.size} values, mean = {s_avg.mean():.3f}")

bin_centers = (bins[:-1] + bins[1:]) / 2

# save the histograms
table = pd.DataFrame({"S_avg": bin_centers})
for (system, chain), c in counts.items():
    table[f"{system}_{chain}"] = c
table.to_csv(os.path.join(out_dir, "s_avg_hist_raw.csv"), index=False)

In [ ]:
# ---- 4. plot ----
plt.rcParams.update({"font.weight": "bold", "axes.labelweight": "bold", "axes.linewidth": 2.5,
                     "mathtext.default": "bf"})
titles = {"sn1": r"$\mathbfit{sn}$-1", "sn2": r"$\mathbfit{sn}$-2"}

fig, axes = plt.subplots(1, 2, figsize=(22, 8), sharey=True)

for ax, chain in zip(axes, chains):
    for system, label in systems.items():
        ax.plot(bin_centers, counts[(system, chain)], "o-", mfc="none", mew=2, ms=7, lw=2,
                color=colors[system], label=label)

    # grey area: where the two distributions overlap
    overlap = np.minimum(counts[("pure_dopc", chain)], counts[("dopc_60", chain)])
    ax.fill_between(bin_centers, overlap, color="0.6", alpha=0.6, lw=0)

    ax.set_title(titles[chain], fontsize=30)
    ax.set_xlabel(r"$\mathbf{\langle S_{CH} \rangle}$", fontsize=28)
    ax.set_xlim(*x_limits)
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(axis="both", which="major", labelsize=26, direction="in", top=True, right=True, length=10, width=2.5)
    ax.tick_params(axis="both", which="minor", direction="in", top=True, right=True, length=5, width=1.8)

axes[0].set_ylabel("Number of chains", fontsize=28)
axes[0].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
axes[0].yaxis.get_offset_text().set_fontsize(22)
axes[1].legend(fontsize=24, frameon=False)

plt.tight_layout(w_pad=4)          # w_pad: space between the two panels
plt.savefig(os.path.join(out_dir, "s_avg_hist_raw.png"), dpi=300)
plt.show()